In [ ]:
import torch
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image
from torch.utils.data import DataLoader

In [3]:
%run "1_data_audit.ipynb"

C:\Users\Lenovo\Desktop\project 2\project2\14_resNet\newData\dataset\train -> True
C:\Users\Lenovo\Desktop\project 2\project2\14_resNet\newData\dataset\clean -> True
C:\Users\Lenovo\Desktop\project 2\project2\14_resNet\newData\datasetv2_TrainClean\train -> True
C:\Users\Lenovo\Desktop\project 2\project2\14_resNet\newData\datasetv2_TrainClean\clean -> True

 newData\dataset\train
Classes: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']

 newData\dataset\clean
Classes: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']

 newData\datasetv2_TrainClean\train
Classes: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']

 newData\datasetv2_TrainClean\clean
Classes: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']
Total number of images: 3313

Number by class:
label
savari       498
kamyun       493
taxi         484
autobus      464
kamyunet     456
minibu

# 2.1 — config and device setup

In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0 # جلوگیری از مشکلات پردازش موازی در نوت‌بوک‌

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

Using device: cpu


# 2.2 — Creating class_to_idx (mapping classes to numbers)

In [6]:
CLASS_NAMES = sorted(inventory_split_df["label"].unique())
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}

print("Class to index mapping:")
for name, idx in CLASS_TO_IDX.items():
    print(f"  {name}: {idx}")

Class to index mapping:
  ambulance: 0
  autobus: 1
  kamyun: 2
  kamyunet: 3
  minibus: 4
  savari: 5
  taxi: 6
  vanet: 7


# 2.3 — train and validation transforms

In [7]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
print("Train transform:")
print(train_transform)
print("\nValidation transform:")
print(val_transform)

Train transform:
Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-10.0, 10.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.8, 1.2), hue=None)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Validation transform:
Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


# 2.4 — Custom dataset class reading

In [8]:
class VehicleDataset(Dataset):
    def __init__(self, dataframe, class_to_idx, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = self.class_to_idx[row["label"]]
        return image, label

# 2.5 — build dataset and dataloaders

In [9]:
train_df = inventory_split_df[inventory_split_df["split"] == "train"].reset_index(drop=True)
val_df = inventory_split_df[inventory_split_df["split"] == "val"].reset_index(drop=True)

train_dataset = VehicleDataset(train_df, CLASS_TO_IDX, transform=train_transform)
val_dataset = VehicleDataset(val_df, CLASS_TO_IDX, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

Train dataset size: 2638
Validation dataset size: 664


# 2.6 — Verifying the validity of a batch

In [10]:
images, labels = next(iter(train_loader))
print(f"\nSample batch image tensor shape: {images.shape}")
print(f"Sample batch label tensor shape: {labels.shape}")
print(f"Label value range: min={labels.min().item()}, max={labels.max().item()}")


Sample batch image tensor shape: torch.Size([32, 3, 224, 224])
Sample batch label tensor shape: torch.Size([32])
Label value range: min=0, max=6


---

# data and model preparation for the `Coarse` model (first stage of the `CASCADE`)

# 3.1 — build the coarse label column (merge kamyun and kamyunet)

In [15]:
MERGE_MAP = {
    "kamyun": "truck_merged",
    "kamyunet": "truck_merged",
}
inventory_split_df["coarse_label"] = inventory_split_df["label"].replace(MERGE_MAP)

print("Original label counts:")
print(inventory_split_df["label"].value_counts())

print("\nCoarse label counts (kamyun + kamyunet merged):")
print(inventory_split_df["coarse_label"].value_counts())

Original label counts:
label
savari       498
kamyun       493
taxi         484
autobus      463
kamyunet     456
minibus      392
ambulance    361
vanet        155
Name: count, dtype: int64

Coarse label counts (kamyun + kamyunet merged):
coarse_label
truck_merged    949
savari          498
taxi            484
autobus         463
minibus         392
ambulance       361
vanet           155
Name: count, dtype: int64


# 3.2 — fixed class_to_idx mapping for the coarse model

In [12]:
COARSE_CLASS_NAMES = sorted(inventory_split_df["coarse_label"].unique())
COARSE_CLASS_TO_IDX = {name: idx for idx, name in enumerate(COARSE_CLASS_NAMES)}
COARSE_IDX_TO_CLASS = {idx: name for name, idx in COARSE_CLASS_TO_IDX.items()}

print("Coarse class to index mapping:")
for name, idx in COARSE_CLASS_TO_IDX.items():
    print(f"  {name}: {idx}")

Coarse class to index mapping:
  ambulance: 0
  autobus: 1
  minibus: 2
  savari: 3
  taxi: 4
  truck_merged: 5
  vanet: 6


# 3.3 — make VehicleDataset flexible to read any label column

In [ ]:
class VehicleDataset(Dataset):
    def __init__(self, dataframe, class_to_idx, label_col="label", transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.label_col = label_col
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = self.class_to_idx[row[self.label_col]]
        return image, label

# 3.4 — build coarse train/val dataset and dataloaders

In [16]:
train_df = inventory_split_df[inventory_split_df["split"] == "train"].reset_index(drop=True)
val_df = inventory_split_df[inventory_split_df["split"] == "val"].reset_index(drop=True)

coarse_train_dataset = VehicleDataset(
    train_df, COARSE_CLASS_TO_IDX, label_col="coarse_label", transform=train_transform
)
coarse_val_dataset = VehicleDataset(
    val_df, COARSE_CLASS_TO_IDX, label_col="coarse_label", transform=val_transform
)
coarse_train_loader = DataLoader(
    coarse_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)
coarse_val_loader = DataLoader(
    coarse_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)
print(f"Coarse train dataset size: {len(coarse_train_dataset)}")
print(f"Coarse validation dataset size: {len(coarse_val_dataset)}")

images, labels = next(iter(coarse_train_loader))
print(f"\nSample batch image shape: {images.shape}")
print(f"Sample batch label shape: {labels.shape}")
print(f"Label value range: min={labels.min().item()}, max={labels.max().item()}")

Coarse train dataset size: 2638
Coarse validation dataset size: 664

Sample batch image shape: torch.Size([32, 3, 224, 224])
Sample batch label shape: torch.Size([32])
Label value range: min=0, max=6
